<a href="https://colab.research.google.com/github/Hadeel-93/HW-8/blob/main/HW_8Q2EXPERIMENT3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import time
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

from torch.utils.data import DataLoader

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True
else:
    print("WARNING: GPU is not available. Training may take much longer.")

In [ ]:
data_path = "/content/data-cifar10/"

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.4914, 0.4822, 0.4465],
        std=[0.2470, 0.2435, 0.2616]
    )
])

train_dataset = torchvision.datasets.CIFAR10(
    root=data_path,
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root=data_path,
    train=False,
    download=True,
    transform=transform
)

print("Training images:", len(train_dataset))
print("Evaluation images:", len(test_dataset))
print("Number of classes:", len(train_dataset.classes))
print("Classes:", train_dataset.classes)

In [ ]:
BATCH_SIZE = 128
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

print("Training batches:", len(train_loader))
print("Evaluation batches:", len(test_loader))

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()

        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False
        )
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.relu = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)

        # Projection shortcut when dimensions change
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = out + identity
        out = self.relu(out)

        return out

In [ ]:
class ResNet10(nn.Module):
    def __init__(self, num_classes=10):
        super(ResNet10, self).__init__()

        self.in_channels = 64

        # Initial convolution and Batch Normalization
        self.conv1 = nn.Conv2d(
            3, 64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)

        # Five stages, two residual blocks per stage
        self.layer1 = self.make_layer(64, 2, stride=1)
        self.layer2 = self.make_layer(128, 2, stride=2)
        self.layer3 = self.make_layer(256, 2, stride=2)
        self.layer4 = self.make_layer(512, 2, stride=2)
        self.layer5 = self.make_layer(512, 2, stride=1)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def make_layer(self, out_channels, num_blocks, stride):
        layers = []

        layers.append(
            ResidualBlock(
                self.in_channels,
                out_channels,
                stride
            )
        )

        self.in_channels = out_channels

        for _ in range(1, num_blocks):
            layers.append(
                ResidualBlock(
                    self.in_channels,
                    out_channels,
                    stride=1
                )
            )

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x

In [ ]:
model = ResNet10(num_classes=10).to(device)

# Count residual blocks
residual_block_count = sum(
    1 for module in model.modules()
    if isinstance(module, ResidualBlock)
)

# Count trainable parameters
total_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print(model)
print("\nNumber of residual blocks:", residual_block_count)
print("Total trainable parameters:", f"{total_params:,}")

assert residual_block_count == 10, "The model must contain exactly 10 residual blocks."

# Test the output dimensions
sample_input = torch.randn(2, 3, 32, 32).to(device)
sample_output = model(sample_input)

print("Output shape:", sample_output.shape)
assert sample_output.shape == (2, 10)

In [ ]:
NUM_EPOCHS = 300
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print("Epochs:", NUM_EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Dropout: Disabled")
print("Batch Normalization: Enabled")

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device, scaler):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs.detach(), 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_accuracy = 100.0 * correct / total

    return epoch_loss, epoch_accuracy

In [ ]:
def evaluate_model(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):
                outputs = model(images)
                loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    evaluation_loss = running_loss / total
    evaluation_accuracy = 100.0 * correct / total

    return evaluation_loss, evaluation_accuracy

In [ ]:
history = {
    "epoch": [],
    "train_loss": [],
    "train_accuracy": [],
    "eval_loss": [],
    "eval_accuracy": []
}

# Automatic mixed precision for GPU training
scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(device.type == "cuda")
)

start_time = time.time()

for epoch in range(NUM_EPOCHS):

    train_loss, train_accuracy = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device,
        scaler
    )

    eval_loss, eval_accuracy = evaluate_model(
        model,
        test_loader,
        criterion,
        device
    )

    history["epoch"].append(epoch + 1)
    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["eval_loss"].append(eval_loss)
    history["eval_accuracy"].append(eval_accuracy)

    print(
        f"Epoch [{epoch + 1}/{NUM_EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.2f}% | "
        f"Eval Loss: {eval_loss:.4f} | "
        f"Eval Acc: {eval_accuracy:.2f}%"
    )

total_training_time = time.time() - start_time

print("\nTraining completed!")
print(f"Total training time: {total_training_time / 60:.2f} minutes")

In [ ]:
final_train_loss = history["train_loss"][-1]
final_train_accuracy = history["train_accuracy"][-1]
final_eval_loss = history["eval_loss"][-1]
final_eval_accuracy = history["eval_accuracy"][-1]

print("===== EXPERIMENT 3: RESNET-10 WITH BATCH NORMALIZATION =====")
print("Epochs:", NUM_EPOCHS)
print(f"Final Training Loss: {final_train_loss:.4f}")
print(f"Final Training Accuracy: {final_train_accuracy:.2f}%")
print(f"Final Evaluation Loss: {final_eval_loss:.4f}")
print(f"Final Evaluation Accuracy: {final_eval_accuracy:.2f}%")
print(f"Total Training Time: {total_training_time / 60:.2f} minutes")
print(f"Trainable Parameters: {total_params:,}")

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(history["epoch"], history["train_loss"], label="Training Loss")
plt.plot(history["epoch"], history["eval_loss"], label="Evaluation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("ResNet-10 with Batch Normalization: Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(history["epoch"], history["train_accuracy"], label="Training Accuracy")
plt.plot(history["epoch"], history["eval_accuracy"], label="Evaluation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("ResNet-10 with Batch Normalization: Accuracy")
plt.legend()
plt.grid(True)
plt.show()